# Realtime Object Tracking + Relative Depth + Rough Metric Distance

Key features are:
- BoTSORT tracking
- Per-frame class counts panel
- Relative depth arrows + %/s rate (to understand if the object is approach or receding) (shows that bounding box increasing/decreasing at "s" % rate)
- Rough metric distance estimate

In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import Counter, defaultdict, deque

model = YOLO("/Users/mldevpilot/prj_ws/my_git_hub_repos/Realtime-Airborne-Object-Detection-and-Tracking/runs/detect/runs/aod_big_run_2/weights/best.pt")
class_names = {0: "aeroplane", 1: "bird", 2: "drone", 3: "helicopter"}

# Colors per class (BGR for OpenCV)
class_colors = {
    0: (255, 100, 0),    # aeroplane — blue
    1: (0, 255, 100),    # bird — green
    2: (0, 0, 255),      # drone — red
    3: (0, 255, 255),    # helicopter — yellow
}

In [2]:
# =====================================================================
# METRIC DEPTH CONFIG — Tune these to our camera and expected objects
# =====================================================================

# Assumed real-world size per class (meters).
# Using the longest dimension (wingspan for planes/birds, rotor diameter for heli, diagonal for drone).
# These are midpoint estimates — adjust if we know the specific objects in our video.
CLASS_REAL_SIZE_M = {
    0: 35.0,   # aeroplane — midsize jet wingspan (~35 m). Cessna would be ~11 m, A380 ~80 m.
    1: 1.0,    # bird — large soaring bird wingspan (~1 m). Small birds ~0.3 m, eagles ~2.2 m.
    2: 0.5,    # drone — consumer quad diagonal (~0.5 m). Tiny: 0.2 m, industrial: 1.5 m.
    3: 14.0,   # helicopter — rotor diameter (~14 m). Small: 10 m, large: 22 m.
}

# Focal length in pixels.
# For a typical smartphone camera (26-28mm equivalent) on a 1080px-wide sensor:
#   focal_px ≈ (focal_mm / sensor_width_mm) × image_width_px
#   ≈ (26 / 36) × 1080 ≈ 780  (for wide-angle phone)
#   ≈ (50 / 36) × 1080 ≈ 1500 (for 50mm equivalent)
# Default 950 is a reasonable middle ground for unknown phone/action cam footage.

FOCAL_LENGTH_PX = 950.0

print("Metric depth config:")
print(f"  Focal length: {FOCAL_LENGTH_PX:.0f} px")
for cls_id, size in CLASS_REAL_SIZE_M.items():
    print(f"  {class_names[cls_id]:12s}: {size:.1f} m assumed size")

Metric depth config:
  Focal length: 950 px
  aeroplane   : 35.0 m assumed size
  bird        : 1.0 m assumed size
  drone       : 0.5 m assumed size
  helicopter  : 14.0 m assumed size


In [3]:
import os

# Input/output paths (absolute, relative to project root)
PROJECT_ROOT = "/Users/mldevpilot/prj_ws/my_git_hub_repos/Realtime-Airborne-Object-Detection-and-Tracking"
input_video = os.path.join(PROJECT_ROOT, "videos/input/video_6.mp4")
output_video = os.path.join(PROJECT_ROOT, "videos/output/video_6.mp4")

os.makedirs(os.path.dirname(output_video), exist_ok=True)

cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise RuntimeError(f"Could not open input video: {input_video}")

fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30  # fall back to 30 if FPS metadata is missing
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Resolution: {width}x{height}  FPS: {fps}  Total frames: {total_frames}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))
if not out.isOpened():
    raise RuntimeError(f"Could not open VideoWriter for: {output_video}")

Resolution: 1900x1080  FPS: 30  Total frames: 2350


In [4]:
# --- Relative depth config  ---
AREA_WINDOW = 15
STABLE_THRESHOLD = 5.0
MIN_SAMPLES = 4


def compute_area_rate(areas, fps):
    """Fit log(area) vs frame index. Returns (rate_%/s, direction) or (None, None)."""
    if len(areas) < MIN_SAMPLES:
        return None, None

    log_areas = np.log(np.array(areas, dtype=np.float64))
    x = np.arange(len(log_areas), dtype=np.float64)

    x_mean = x.mean()
    y_mean = log_areas.mean()
    slope = ((x - x_mean) * (log_areas - y_mean)).sum() / ((x - x_mean) ** 2).sum()

    rate_pct_per_sec = slope * fps * 100.0

    if rate_pct_per_sec > STABLE_THRESHOLD:
        direction = "approaching"
    elif rate_pct_per_sec < -STABLE_THRESHOLD:
        direction = "receding"
    else:
        direction = "stable"

    return rate_pct_per_sec, direction


def compute_metric_distance(bbox_w, bbox_h, cls_id):
    """
    Estimate distance in meters using pinhole camera model.
    Uses max(w, h) as the bbox size to approximate the longest visible span.
    """
    bbox_size = max(bbox_w, bbox_h)
    if bbox_size < 1:  # avoid division by zero for degenerate boxes
        return None

    real_size = CLASS_REAL_SIZE_M.get(cls_id, 1.0)
    distance = (real_size * FOCAL_LENGTH_PX) / bbox_size
    return distance


ARROW = {"approaching": "^", "receding": "v", "stable": "~"}

In [5]:
def draw_count_panel(frame, counts, class_names, class_colors):
    """Draw a semi-transparent panel in the top-left with per-class counts."""
    pad = 10
    line_h = 28
    panel_w = 220
    panel_h = pad * 2 + line_h * (len(class_names) + 1)
    x0, y0 = 10, 10

    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + panel_w, y0 + panel_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    cv2.putText(
        frame, "In this frame:",
        (x0 + pad, y0 + pad + 18),
        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA,
    )

    for i, cls_id in enumerate(sorted(class_names.keys())):
        name = class_names[cls_id]
        count = counts.get(cls_id, 0)
        color = class_colors[cls_id]
        y = y0 + pad + 18 + line_h * (i + 1)

        cv2.rectangle(frame, (x0 + pad, y - 14), (x0 + pad + 14, y), color, -1)
        text = f"{name:11s} {count}"
        cv2.putText(
            frame, text,
            (x0 + pad + 22, y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA,
        )

    return frame


def draw_depth_label(frame, x1, y1, x2, y2, cls_id, obj_id, rate, direction, distance_m):
    """
    Draw the depth annotation below the bounding box.
    Shows: arrow + rate + estimated distance.
    Example: "#19 ^ +14%/s ~180m"
    """
    # Build the label string
    parts = [f"#{obj_id}"]

    # Relative depth part
    if direction is not None:
        arrow = ARROW[direction]
        parts.append(f"{arrow} {rate:+.0f}%/s")
    else:
        parts.append("--")

    # Metric distance part
    if distance_m is not None:
        if distance_m >= 1000:
            parts.append(f"~{distance_m / 1000:.1f}km")
        else:
            parts.append(f"~{distance_m:.0f}m")

    label = " ".join(parts)

    # Position: just below the bottom-left of the bbox
    label_x = int(x1)
    label_y = int(y2) + 18

    # Color based on direction
    if direction == "approaching":
        color = (0, 140, 255)   # orange
    elif direction == "receding":
        color = (255, 200, 0)   # cyan
    else:
        color = (200, 200, 200) # grey

    cv2.putText(
        frame, label,
        (label_x, label_y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA,
    )
    return frame

In [6]:
# --- State ---
track_areas = defaultdict(lambda: deque(maxlen=AREA_WINDOW))

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run BoTSORT tracking
    results = model.track(
        frame,
        imgsz=640,
        conf=0.25,
        tracker="botsort.yaml",
        persist=True,
        verbose=False,
    )

    # Draw default boxes + IDs
    annotated_frame = results[0].plot()

    # --- Per-frame class counts ---
    boxes = results[0].boxes
    if boxes is not None and boxes.cls is not None and len(boxes.cls) > 0:
        classes_this_frame = boxes.cls.int().tolist()
        frame_counts = Counter(classes_this_frame)
    else:
        frame_counts = Counter()

    # --- Relative depth + Metric distance ---
    if boxes is not None and boxes.id is not None:
        ids = boxes.id.int().tolist()
        classes = boxes.cls.int().tolist()
        xyxy = boxes.xyxy.tolist()

        for obj_id, cls_id, box in zip(ids, classes, xyxy):
            x1, y1, x2, y2 = box
            bbox_w = x2 - x1
            bbox_h = y2 - y1
            area = bbox_w * bbox_h

            # --- Relative depth (from notebook 8) ---
            history = track_areas[obj_id]
            if len(history) > 0 and (frame_count - history[-1][0]) > AREA_WINDOW:
                history.clear()
            history.append((frame_count, area))

            recent_areas = [a for (f, a) in history if (frame_count - f) <= AREA_WINDOW and a > 0]
            rate, direction = compute_area_rate(recent_areas, fps)

            # --- Metric distance (NEW) ---
            distance_m = compute_metric_distance(bbox_w, bbox_h, cls_id)

            # Draw combined label
            annotated_frame = draw_depth_label(
                annotated_frame, x1, y1, x2, y2, cls_id, obj_id, rate, direction, distance_m
            )

    # Overlay count panel
    annotated_frame = draw_count_panel(
        annotated_frame, frame_counts, class_names, class_colors
    )

    out.write(annotated_frame)

    frame_count += 1
    if frame_count % 100 == 0:
        print(f"Processed {frame_count}/{total_frames} frames")

cap.release()
out.release()

print(f"\nDone! Output saved to: {output_video}")

Processed 100/2350 frames
Processed 200/2350 frames
Processed 300/2350 frames
Processed 400/2350 frames
Processed 500/2350 frames
Processed 600/2350 frames
Processed 700/2350 frames
Processed 800/2350 frames
Processed 900/2350 frames
Processed 1000/2350 frames
Processed 1100/2350 frames
Processed 1200/2350 frames
Processed 1300/2350 frames
Processed 1400/2350 frames
Processed 1500/2350 frames
Processed 1600/2350 frames
Processed 1700/2350 frames
Processed 1800/2350 frames
Processed 1900/2350 frames
Processed 2000/2350 frames
Processed 2100/2350 frames
Processed 2200/2350 frames
Processed 2300/2350 frames

Done! Output saved to: /Users/mldevpilot/prj_ws/my_git_hub_repos/Realtime-Airborne-Object-Detection-and-Tracking/videos/output/video_6.mp4


## The formula

```
distance_m = (real_size_m × focal_length_px) / bbox_size_px
```

Where:
- `real_size_m` — assumed physical size of the object (wingspan, rotor diameter, etc.)
- `focal_length_px` — camera focal length in pixel units
- `bbox_size_px` — `max(bbox_width, bbox_height)` in the current frame

## Accuracy disclaimer

- **Rough metric distance estimate** (in meters) per detected object, using the pinhole camera model and assumed real-world object sizes.

This gives an **order-of-magnitude** estimate (±50–300%) because:
- We don't know the exact aircraft type (Cessna vs A380 have very different wingspans)
- We assume a fixed focal length (may not match the actual camera)
- Objects at oblique angles show a foreshortened projection

Good enough for: "is this at ~100 m or ~2 km?". Not good for precision ranging.

### `CLASS_REAL_SIZE_M` — assumed physical sizes

We assign a single midpoint size per class:

| Class | Assumed size | Represents |
|-------|-------------|------------|
| Aeroplane | 35 m | Midsize commercial jet (Boeing 737 wingspan) |
| Bird | 1.0 m | Large soaring bird (hawk/eagle) |
| Drone | 0.5 m | Consumer quadcopter (DJI Mavic class) |
| Helicopter | 14 m | Medium utility helicopter rotor diameter |

**This is the biggest source of error.** If the actual plane is a Cessna 172 (wingspan 11 m) but we assume 35 m, our distance estimate will be ~3x too high. There's no way around this without aircraft identification.

### `FOCAL_LENGTH_PX` — camera intrinsic

Default: 950 px. This assumes a typical smartphone camera (~28mm equivalent) capturing at 1080px width.

To compute for our specific camera:
```
focal_px = (focal_mm / sensor_width_mm) × image_width_px
```

### Display format

Below each bbox we now see:
```
#19 ^ +14%/s ~180m
```
- `#19` — track ID
- `^ +14%/s` — relative depth (approaching at 14% area growth/s)
- `~180m` — estimated absolute distance

For distances ≥1000 m, it shows km: `~1.2km`

### How to improve accuracy

1. **Know your camera** — set `FOCAL_LENGTH_PX` precisely using EXIF data or manufacturer specs.
2. **Add aircraft-type subclasses** — fine-tune your YOLO model to distinguish Cessna vs 737 vs A380, then assign per-subclass sizes.
3. **Cross-validate with radar/ADS-B** — if we have ground truth, calibrate the constants.